# Ch 28 — Seq2seq 미니 — ITN

Encoder-Decoder (T5/byT5) 구조로 ITN (Inverse Text Normalization) 모델 학습. "공일공" → "010", "이천이십육년" → "2026년". STT 후처리 직결.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch28_seq2seq_itn.ipynb)

In [ ]:
# !pip install -q transformers datasets accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 컨셉 — 세 번째 모양

| 구조 | encoder | decoder | 적합 작업 |
|---|---|---|---|
| **Encoder-only** (BERT) | O | X | 분류·NER (Ch 25) |
| **Decoder-only** (GPT) | X | O | 생성 (Ch 24) |
| **Encoder-Decoder** (T5) | O | O | **변환** (번역·요약·ITN) |

```
입력 → [encoder] → context vectors
                       | (cross-attention)
           [decoder] → 출력 (autoregressive)
```

**한국어 ITN 추천**: **byT5-small** — byte-level 이라 한국어/숫자/한자 OOV 없음.

## 5. 데이터 합성 — 1만 페어 (최소 예제)

In [ ]:
# itn_synth.py
import random, json

NUMBERS_KO = {0:"영", 1:"일", 2:"이", 3:"삼", 4:"사",
              5:"오", 6:"육", 7:"칠", 8:"팔", 9:"구"}

def num_to_ko(n):
    """숫자를 한국어 낱자 나열로 (예: 123 → '일이삼')."""
    return "".join(NUMBERS_KO[int(d)] for d in str(n))

def num_to_ko_year(y):
    """연도를 한국어 읽기로 (예: 2026 → '이천이십육')."""
    result = ""
    if y >= 1000:
        t = y // 1000
        result += ("" if t == 1 else NUMBERS_KO[t]) + "천"
        y %= 1000
    if y >= 100:
        h = y // 100
        result += ("" if h == 1 else NUMBERS_KO[h]) + "백"
        y %= 100
    if y >= 10:
        d = y // 10
        result += ("" if d == 1 else NUMBERS_KO[d]) + "십"
        y %= 10
    if y > 0:
        result += NUMBERS_KO[y]
    return result

def ko_money(v):
    """금액을 한국어 단위로 (간단 버전)."""
    if v >= 10000:
        man = v // 10000
        rest = v % 10000
        result = num_to_ko_year(man) + "만"
        if rest > 0:
            result += num_to_ko_year(rest)
        return result + "원"
    return num_to_ko_year(v) + "원"

def gen_pair():
    kind = random.choice(["phone", "year", "money", "percent"])
    if kind == "phone":
        digits = "010" + "".join(random.choices("0123456789", k=8))
        spoken = (num_to_ko(int(digits[:3])) + " " +
                  num_to_ko(int(digits[3:7])) + " " +
                  num_to_ko(int(digits[7:])))
        written = digits[:3] + "-" + digits[3:7] + "-" + digits[7:]
    elif kind == "year":
        y = random.randint(1980, 2099)
        spoken = num_to_ko_year(y) + "년"
        written = f"{y}년"
    elif kind == "money":
        v = random.choice([10000, 14000, 50000, 1500000, 30000, 200000])
        spoken = ko_money(v)
        written = f"{v}원"
    elif kind == "percent":
        p = random.randint(1, 99)
        spoken = num_to_ko_year(p) + "퍼센트"
        written = f"{p}%"
    return spoken, written

# 테스트
for _ in range(5):
    s, w = gen_pair()
    print(f"  '{s}' → '{w}'")    

In [ ]:
# 1만 페어 합성 (노트북 시연: 1,000개)
N = 1000
pairs = [gen_pair() for _ in range(N)]

with open("itn_train.jsonl", "w") as f:
    for sp, wr in pairs:
        f.write(json.dumps({"input": sp, "output": wr}, ensure_ascii=False) + "\n")

print(f"ITN 페어 {N}개 저장")
print("샘플:")
for sp, wr in pairs[:5]:
    print(f"  '{sp}' → '{wr}'")

## 6. 학습 — byT5-small fine-tune (실전)

In [ ]:
# itn_train.py
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer,
                           DataCollatorForSeq2Seq)
from datasets import load_dataset

base = "google/byt5-small"
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForSeq2SeqLM.from_pretrained(base)
model = model.to(device)

ds = load_dataset("json", data_files="itn_train.jsonl")["train"]

def fmt(b):
    enc = tok(b["input"], max_length=128, truncation=True, padding="max_length")
    with tok.as_target_tokenizer():
        lab = tok(b["output"], max_length=128, truncation=True, padding="max_length")
    enc["labels"] = lab["input_ids"]
    return enc

ds = ds.map(fmt, batched=True).remove_columns(["input", "output"])

args = Seq2SeqTrainingArguments(
    output_dir="itn_out", num_train_epochs=5,
    learning_rate=3e-4, per_device_train_batch_size=16,
    warmup_steps=100, lr_scheduler_type="linear",
    bf16=(device == "cuda"), predict_with_generate=True,
    logging_steps=50, save_steps=500,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=ds,
    tokenizer=tok,
    data_collator=DataCollatorForSeq2Seq(tok, model=model)
)
trainer.train()
trainer.save_model("itn_out/final")

## 7. 추론 + 평가

In [ ]:
# itn_infer.py
from transformers import pipeline

itn = pipeline("text2text-generation", model="itn_out/final")

test_cases = [
    ("공일공 일이삼사 오육칠팔", "010-1234-5678"),
    ("이천이십육년", "2026년"),
    ("오만원", "50000원"),
    ("칠퍼센트", "7%"),
]

print("=== ITN 추론 결과 ===")
correct = 0
for spoken, expected in test_cases:
    pred = itn(spoken)[0]["generated_text"]
    ok = pred == expected
    if ok: correct += 1
    print(f"  입력: '{spoken}'")
    print(f"  예상: '{expected}' | 예측: '{pred}' {'O' if ok else 'X'}")
    print()

print(f"Exact Match: {correct}/{len(test_cases)} = {correct/len(test_cases):.0%}")

In [ ]:
# 전체 검증셋 EM 평가
# 학습셋의 20%를 검증용으로 별도 분리하는 게 이상적 (여기서는 간단 버전)
val_pairs = [gen_pair() for _ in range(100)]

correct = 0
for sp, wr in val_pairs:
    pred = itn(sp)[0]["generated_text"]
    if pred == wr: correct += 1

print(f"검증 EM: {correct}/{len(val_pairs)} = {correct/len(val_pairs):.1%}")
print("(1만 페어 학습 시 EM 88~95% 예상)")

## Part 7 마무리

| 챕터 | 무엇을 |
|---|---|
| Ch 22 | 기성 sLLM 5종 비교 + 결정 트리 |
| Ch 23 | from-scratch vs FT — 노트북 메모리 산수 |
| Ch 24 | LoRA · QLoRA — `peft` 30줄 |
| Ch 25 | Encoder NER — 도메인 entity 추출 |
| Ch 26 | Decoder LoRA + 추가 사전학습 |
| Ch 27 | Distillation 미니 — Teacher → Student |
| **Ch 28** | **Seq2seq 미니 — ITN** |

다음 단계 → Part 8 프로덕션 운영.

## 연습문제

1. byT5-small ITN 1만 페어 학습. EM 측정.
2. **decoder-only LoRA** (Qwen 0.5B) 로 같은 데이터 학습. seq2seq 와 EM 비교.
3. 합성 데이터 1K / 5K / 10K / 50K 학습 시 EM 곡선.
4. byT5-small vs t5-small (영어) — 한국어 ITN 에서 차이.
5. **(생각해볼 것)** ITN 외에 본인 도메인에 seq2seq 가 적합한 작업은? STT 후처리, 번역 외에.

In [ ]:
# 연습 1: 1만 페어 학습 + EM
# pairs_10k = [gen_pair() for _ in range(10000)]
# ... 학습 반복


In [ ]:
# 연습 2: decoder-only LoRA (Qwen 0.5B) vs seq2seq 비교
# instruction 형식: "다음 음성 표기를 숫자/기호로 변환: 공일공 일이삼사 오육칠팔 → "


In [ ]:
# 연습 3: 데이터 크기별 EM 곡선
for n in [1000, 5000, 10000]:
    print(f"{n}개 학습 → EM: (학습 후 측정)")


In [ ]:
# 연습 4: byT5-small vs t5-small 비교
# base_byt5 = "google/byt5-small"    # byte-level
# base_t5   = "t5-small"             # sentencepiece (영어 위주)
